# HYCOM Reanalysis Download

Downloads HYCOM GLBv0.08/expt_53.X reanalysis (SSH, U, V) for the equatorial Pacific:

| Parameter | Value |
|-----------|-------|
| Longitude | 180–260 °E |
| Latitude  | 10 °S – 10 °N |
| Depth     | 0–500 m |
| Variables | SSH, U, V |
| Period    | 01 Sep 2012 – 31 Dec 2013 |

Files are saved monthly to `hycom_data/`.  No authentication is required.

**Server truncation note:** The HYCOM THREDDS OPeNDAP server silently fills
values with zeros when a single request exceeds its size limit (~1–2 GB).
The download function uses `chunk_days=2` (≈0.9 GB/chunk for U+V) to stay
well under this limit.  Delete any existing monthly files before re-downloading
to force a fresh fetch — the script skips files that already exist.

In [ ]:
import sys
from pathlib import Path

# Add forecasts/ to path so hycom_download is importable from anywhere
sys.path.insert(0, str(Path("__file__").resolve().parent))

from forecasts.hycom_download import download_hycom

In [ ]:
# ── Test download: 1 week, same region ──────────────────────────────────────
# Estimated size: ~200 MB (well under 1 GB)
test_files = download_hycom(
    lon_min=180,
    lon_max=260,
    lat_min=-10,
    lat_max=10,
    depth_min=0,
    depth_max=-500,
    variables=["SSH", "U", "V"],
    time_start="09-01-2012",
    time_end="09-07-2012",
    output_dir="/data/SO3/edavenport/hycom_data",
)

In [ ]:
import xarray as xr

# Sanity check on test file
if test_files:
    ds = xr.open_dataset(test_files[0])
    print(ds)
    ds.close()

In [ ]:
# ── Full download: Sep 2012 – Dec 2013 ──────────────────────────────────────
# chunk_days=2 keeps each OPeNDAP request under ~1 GB to avoid server truncation.
# Delete any corrupt/incomplete monthly files before running so they get re-fetched.
OUTPUT_DIR = "/data/SO3/edavenport/tpose6/hycom_data"

files = download_hycom(
    lon_min=180,
    lon_max=260,
    lat_min=-10,
    lat_max=10,
    depth_min=0,
    depth_max=-500,
    variables=["SSH", "U", "V"],
    time_start="09-01-2012",
    time_end="12-31-2013",
    output_dir=OUTPUT_DIR,
    chunk_days=2,
)

In [ ]:
print(f"Downloaded {len(files)} file(s):")
for f in files:
    size_mb = Path(f).stat().st_size / 1e6 if Path(f).exists() else 0
    print(f"  {f.name}  ({size_mb:.1f} MB)")